# **Purpose of Notebook**
This notebook assesses the precision of *ExtractNames* function.

In [4]:
origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'

import sys
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')

In [5]:
#Set up Environment
import os,cv2
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Extract_Data')
from Split import HoriPart, VertPart
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
from OCR import Clova
api_url='https://1srlcnzf08.apigw.ntruss.com/custom/v1/2412/9a859f9b3a7d952aad17f388d1a445041f8aef0f1eccc6fcce8d3a856272fcbd/general'
secret_key='eEhyV0NGRlRLVXpZVWZnWFNDamRVaWFYZ1NSQ1NKSFI='


os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [19]:
import json, os, cv2
import pandas as pd

Year,Showa='1942','17'
Quality='HQ'

df = pd.read_csv(origin+'/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

# Experiment

In [28]:
StepAError,StepBError,StepCError,MainError=[],[],[],[]
Data={}
project='Off'
PageList=range(8,119,1)
for Page in PageList:
    #Step A
    try:
        file_path2='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.jpg'
        img=cv2.imread(os.path.join(origin,file_path2))
        try:
            if img==None:
                file_path2='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.png'
                img=cv2.imread(os.path.join(origin,file_path2))    
        except:
            img=cv2.imread(os.path.join(origin,file_path2))    
        
        
        #Convert to book page
        BookPage=2*Page-14
        Rows=df[(df['Page']==BookPage)]
        NxRows=df[(df['Page']==BookPage+1)]
        if len(Rows)==0:
            print('No Office at Right Side. Page'+str(BookPage))
        if len(NxRows)==0:
            print('No Office at Left Side. Page'+str(BookPage+1))
                
        texts=Vision(img,'zh',client)
        textsCLOVA=Clova(origin,Page,api_url,secret_key,Year,Quality)
    except:
        StepAError.append(Page)
        continue
        
    
    #Step B
    try:
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')
        from Split import VertPart
        
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from Organize import Filter, ConvertDict
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
        from PageCut import HoriPart
        
        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.jpg'
        path=os.path.join(origin,file_path)
        
        HoriPoint=HoriPart(img,Page,texts)[0]
        
        HH,WW=img.shape[:2]
        VertPoint=1*HH//3
        
        ##Right Page
        BoxR=Filter(BookPage,texts,HoriPoint)
        BoxR=ConvertDict(BoxR)
        
        ##Left Page
        BoxL=Filter(BookPage+1,texts,HoriPoint)
        BoxL=ConvertDict(BoxL)

        Dict={}
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from Organize import FilterBox
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from DetectOffice import DetectOffice
        LocList=[]
        
        #RightPage
        OfficeList=Rows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxR, Office,VertPoint))
            BoxR=FilterBox(BoxR,LocList,VertPoint)
        
        #LeftPage
        OfficeList=NxRows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxL, Office,VertPoint))
            BoxL=FilterBox(BoxL,LocList,VertPoint)
        
        Dict[Page]=LocList
    except:
        StepBError.append(Page)
        continue
    
    #Step C
    try:
        import itertools
        PosiDict,PosiDict_CLOVA={},{}
        
        #Split quardrant into offices and detect Positions
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Position')
        from OrganizePosi import Split, SplitClova, Crop, CropClova
        from DetectPosi import DetectPosi,DetectPosiClova,AggregatePosi
        
        CrossWalk=pd.read_csv(origin+"Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
        Positions=CrossWalk.tolist()
        PosiDict['Pre']=[]
        PosiDict_CLOVA['Pre']=[]
        
        DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)
        DF_CLOVA=CropClova(textsCLOVA,HoriPoint,VertPoint,Dict,Page)
        
        ##UR
        BoxList,BoxListCLOVA=Split(DF['UR']['Box'],DF['UR']['Thres']),SplitClova(DF_CLOVA['UR']['Box'],DF_CLOVA['UR']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UR']['Thres'],PosiDict_CLOVA,Positions)
        
        ##LR
        BoxList,BoxListCLOVA=Split(DF['LR']['Box'],DF['LR']['Thres']),SplitClova(DF_CLOVA['LR']['Box'],DF_CLOVA['LR']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LR']['Thres'],PosiDict_CLOVA,Positions)
        
        ##UL
        BoxList,BoxListCLOVA=Split(DF['UL']['Box'][1:],DF['UL']['Thres']),SplitClova(DF_CLOVA['UL']['Box'],DF_CLOVA['UL']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UL']['Thres'],PosiDict_CLOVA,Positions)
        
        #LL
        BoxList,BoxListCLOVA=Split(DF['LL']['Box'],DF['LL']['Thres']),SplitClova(DF_CLOVA['LL']['Box'],DF_CLOVA['LL']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LL']['Thres'],PosiDict_CLOVA,Positions)
        
        FinalPosiDict=AggregatePosi(PosiDict,PosiDict_CLOVA)
    
    
        from DetectPosi import RefPosiDict
        FinalPosiDict=RefPosiDict(texts,FinalPosiDict)
        
        from Layout import RefineVert
        VertPoint2=RefineVert(img,LocList,FinalPosiDict,VertPoint,HoriPoint)
        
        from DetectPosi import RefPosiDict2
        FinalPosiDict=RefPosiDict2(FinalPosiDict,VertPoint,VertPoint2)
        FinalPosiDict={Office: list(itertools.chain(*FinalPosiDict[Office])) for Office in PosiDict}

    except:
        StepCError.append(Page)
        continue
        
    #MainCode
    try:        
        from OrganizeName import FilterPosi, RenewPosiList,GetOffice
        from Extract import ExtractName
    
        DF=Crop(texts,HoriPoint,VertPoint2,Dict,Page)
        Data[str(Page)]=[]
        
        #UpperRight
        Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'UR','NA',img,project,origin,Year,Page))
        #LowerRight
        Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'LR','UR',img,project,origin,Year,Page))
        #BottomLeft
        Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'UL','LR',img,project,origin,Year,Page))
        #BottomLeft
        Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'LL','UL',img,project,origin,Year,Page))
    
    except:
        MainError.append(Page)
        continue

No Office at Right Side. Page2
PrePosi
守衛長
Failed to find tables
Failed to detect top rows
監視
Failed to find tables
監督
主事
書記
Failed to find tables
Failed to find tables
Failed to find tables
主事
書記
雇
Failed to find tables
囑託員
Failed to find tables
PrePosi
Horizontal Line was not automatically detected... Defining line arbitrariry...
PrePosi
Failed to find tables
書記
雇
Failed to detect top rows
講師
Failed to find tables
囑託員
Failed to find tables
主事
主事
書記
技手
雇
主事
主事
Failed to detect top rows
技師
Failed to find tables
技師
書記
雇
Failed to find tables
主事
Failed to detect top rows
技師
Failed to find tables
技師
技師
書記
技手
No Office at Left Side. Page9
Horizontal Line was not automatically detected... Defining line arbitrariry...
PrePosi
囑託員
主事
雇
Failed to detect top rows
Failed to find tables
書記
書記
PrePosi
Failed to detect top rows
Failed to find tables
Horizontal Line was not automatically detected... Defining line arbitrariry...
PrePosi
雇
主事
書記
書記
守衛長
Failed to find tables
雇
主事
書記
雇
書記
雇
主事
Failed to

In [29]:
MainError

[39, 56, 57]

In [30]:
ManList,Eng1List,ClerkList,WorkList,Eng2List=[],[],[],[],[]
for PageNum in PageList:
    try:    
        Page = Data[str(PageNum)]
        for Quadrant in Page:
            for Office in Quadrant.keys():
                for Posi in Quadrant[Office]:
                    Position=Quadrant[Office][Posi]
                    if Posi=='主事':
                        ManList.append(Position)
                    if Posi=='技師':
                        Eng1List.append(Position)
                    if Posi=='書記':
                        ClerkList.append(Position)
                    if Posi=='雇':
                        WorkList.append(Position)
                    if Posi=='技手':
                        Eng2List.append(Position)
    except:
        print('Failed at Page'+str(PageNum))    

Failed at Page9
Failed at Page12
Failed at Page16
Failed at Page17
Failed at Page18
Failed at Page19
Failed at Page23
Failed at Page26
Failed at Page30
Failed at Page34
Failed at Page35
Failed at Page37
Failed at Page41
Failed at Page47
Failed at Page49
Failed at Page54
Failed at Page55
Failed at Page58
Failed at Page63
Failed at Page64
Failed at Page65
Failed at Page66
Failed at Page69
Failed at Page70
Failed at Page72
Failed at Page73
Failed at Page74
Failed at Page76
Failed at Page77
Failed at Page78
Failed at Page79
Failed at Page80
Failed at Page82
Failed at Page86
Failed at Page97
Failed at Page100
Failed at Page114
Failed at Page118
Failed at Page119
Failed at Page120
Failed at Page121
Failed at Page122
Failed at Page123
Failed at Page124
Failed at Page125


# Summary of performance

**1. Non-Running Error**

- Step-Wise Error

In [31]:
def ErrorRate(ErrorList,PageList):
    return(round(len(ErrorList)/len(PageList),2))

In [32]:
ErrorRate(StepAError,PageList),ErrorRate(StepBError,PageList),ErrorRate(StepCError,PageList),ErrorRate(MainError,PageList)

(0.05, 0.06, 0.27, 0.03)

In [33]:
MainError

[39, 56, 57]

**2. Performance by postion**

In [34]:
def Accuracy(List):
    TotalNum=len(List)
    SuccessNum=len([d for d in List if d!=[]])
    return(round(SuccessNum/TotalNum,2))

In [35]:
{'主事':Accuracy(ManList),
 '技師':Accuracy(Eng1List),
 '書記':Accuracy(ClerkList),
 '技手':Accuracy(Eng2List),
 '雇':Accuracy(WorkList)}

{'主事': 0.94, '技師': 0.87, '書記': 0.8, '技手': 0.71, '雇': 0.56}

# Convert data to tables

In [36]:
from OrganizeName import Convert2
DF=pd.DataFrame(columns=['Office','Name'])
for PageNum in PageList:
    try:    
        Page = Data[str(PageNum)]
        for Quadrant in Page:
            for Office in list(Quadrant.keys()):
                for Posi in list(Quadrant[Office].keys()):
                    if len(Quadrant[Office][Posi])==0:
                        continue
                    Table=Quadrant[Office][Posi][0]['Table']
                    Names=Quadrant[Office][Posi][0]['Names']
                    
                    
                    NewData=Convert2(PageNum,Office,Posi,Table,Names)
                    DF=pd.concat([DF,NewData])
    except:
        continue
                    


In [37]:
DF.to_csv(origin+'/Tokyo_Jobs/Processed_Data/'+str(Year)+'/DataFrame/Data.csv', index=False)  

In [20]:
DF=pd.read_csv(origin+'/Tokyo_Jobs/Processed_Data/'+str(Year)+'/DataFrame/Data.csv')

In [9]:
def GetBorderOffice(Page):
    BookPage=2*Page-14
    OfficeList=list(df['Office'][df['Page']==BookPage-1])
    while len(OfficeList)==0:
        BookPage=BookPage-1
        OfficeList=list(df['Office'][df['Page']==BookPage-1])
        if BookPage<0:
            break
            
    if len(OfficeList)==0:
        return(None)
    elif len(OfficeList)>1:
        LastOffice=OfficeList[-1]
        return(LastOffice)
    elif len(OfficeList)==0:
        LastOffice=OfficeList[0]
        return(LastOffice)
        

def ConvertPage(BookPage):
    if BookPage%2==0:
        Page=0.5*BookPage+7
    elif BookPage%2==1:
        Page=0.5*(BookPage-1)+7
    return(int(Page))
        

In [21]:
import numpy as np

mask = (DF['Office'] == 'Pre')
DF_valid = DF[mask]
DF.loc[mask, 'Office'] = DF_valid['Page'].apply(GetBorderOffice)
DF=DF[DF['Office']!=None]

In [22]:
df['BookPage']=df['Page']
df['Page']=df['BookPage'].apply(ConvertPage)

In [23]:
DF=pd.merge(DF,df, on=['Office','Page'], how='left')

In [26]:
FinDF=DF[['Dept','Office','Name','Page','Total','SubOffice','Posi']]

In [27]:
FinDF.to_csv(origin+'/Tokyo_Jobs/Processed_Data/'+str(Year)+'/DataFrame/FinData.csv', index=False)  